# 08 — Model Evaluation

Comprehensive evaluation: metrics, actual vs predicted, residual analysis, error breakdown by segment.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats
import joblib, yaml, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.facecolor':'#0f172a','axes.facecolor':'#1e293b',
    'axes.edgecolor':'#334155','axes.labelcolor':'#e2e8f0','xtick.color':'#94a3b8',
    'ytick.color':'#94a3b8','text.color':'#e2e8f0','grid.color':'#334155',
    'figure.titlesize':16,'axes.titlesize':13})
PALETTE = ['#38bdf8','#fb7185','#34d399','#fbbf24','#a78bfa','#f97316']

with open('../configs/paths.yaml') as f:
    paths = yaml.safe_load(f)
with open('../configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

In [ ]:
# ── Load data, model, and predictions ────────────────────────────────────────
df_raw = pd.read_csv(f'../{paths["data"]["raw"]}')
df_eng = pd.read_csv(f'../{paths["data"]["processed"]}data_engineered.csv')

TARGET = cfg['project']['target']
TEST_SIZE = cfg['project']['test_size']
X = df_eng.drop(columns=[TARGET])
y = df_eng[TARGET]
split_idx = int(len(X) * (1 - TEST_SIZE))

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

model = joblib.load(f'../{paths["artifacts"]["models"]}tuned_model.pkl')
model.fit(X_train, y_train)  # re-fit in case kernel restarted

y_pred = model.predict(X_test)
residuals = y_test.values - y_pred

print(f'Test samples: {len(y_test)}')
print(f'R²  : {r2_score(y_test, y_pred):.4f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}')
print(f'MAE : {mean_absolute_error(y_test, y_pred):.2f}')
mape = np.mean(np.abs(residuals / y_test.values)) * 100
print(f'MAPE: {mape:.2f}%')

In [ ]:
# ── Actual vs Predicted ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Model Evaluation — Actual vs Predicted')

# Scatter
axes[0].scatter(y_test, y_pred, alpha=0.6, color=PALETTE[0], s=30)
lo, hi = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
axes[0].plot([lo,hi],[lo,hi],'r--',linewidth=2,label='Perfect prediction')
axes[0].set(title='Scatter: Actual vs Predicted', xlabel='Actual cnt', ylabel='Predicted cnt')
axes[0].legend(); axes[0].grid(alpha=0.2)

# Time series overlay
test_dates = pd.to_datetime(df_raw['dteday'].iloc[split_idx:].values, dayfirst=True)
axes[1].plot(test_dates, y_test.values, color=PALETTE[0], linewidth=1.5, label='Actual', alpha=0.9)
axes[1].plot(test_dates, y_pred, color=PALETTE[1], linewidth=1.5, label='Predicted', linestyle='--')
axes[1].set(title='Test Period: Actual vs Predicted', xlabel='Date', ylabel='cnt')
axes[1].legend(); axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.savefig('../outputs/eval_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Residual Analysis ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Residual Analysis')

# Residuals vs Predicted
axes[0].scatter(y_pred, residuals, alpha=0.5, color=PALETTE[0], s=25)
axes[0].axhline(0, color='white', linestyle='--', linewidth=2)
axes[0].set(title='Residuals vs Predicted', xlabel='Predicted', ylabel='Residual (Actual − Predicted)')
axes[0].grid(alpha=0.2)

# Histogram of residuals
axes[1].hist(residuals, bins=25, color=PALETTE[2], edgecolor='#0f172a', alpha=0.9)
axes[1].axvline(0, color='white', linestyle='--', linewidth=2)
axes[1].set(title='Residual Distribution', xlabel='Residual', ylabel='Frequency')
axes[1].grid(axis='y', alpha=0.3)

# Q-Q plot
stats.probplot(residuals, plot=axes[2])
axes[2].get_lines()[0].set(markerfacecolor=PALETTE[0], markersize=4, alpha=0.6)
axes[2].get_lines()[1].set(color=PALETTE[1], linewidth=2)
axes[2].set_title('Q-Q Plot of Residuals')
axes[2].grid(alpha=0.2)

plt.tight_layout()
plt.savefig('../outputs/residual_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Error breakdown by season / weather ───────────────────────────────────────
test_meta = df_raw.iloc[split_idx:][['season','weathersit','mnth','yr','workingday']].copy()
test_meta['abs_error'] = np.abs(residuals)
test_meta['pred'] = y_pred
test_meta['actual'] = y_test.values

test_meta['season_label']  = test_meta['season'].map({1:'Spring',2:'Summer',3:'Fall',4:'Winter'})
test_meta['weather_label'] = test_meta['weathersit'].map({1:'Clear',2:'Mist',3:'Light Rain',4:'Heavy Rain'})

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Mean Absolute Error by Segment')

for ax, col, order, title in [
    (axes[0], 'season_label',  ['Spring','Summer','Fall','Winter'],    'By Season'),
    (axes[1], 'weather_label', ['Clear','Mist','Light Rain','Heavy Rain'], 'By Weather'),
    (axes[2], 'mnth',          None,                                   'By Month'),
]:
    if col == 'mnth':
        seg = test_meta.groupby(col)['abs_error'].mean()
        ax.bar(seg.index.astype(str), seg.values, color=PALETTE[0], edgecolor='#0f172a')
    else:
        seg = test_meta.groupby(col)['abs_error'].mean()
        if order:
            seg = seg.reindex([o for o in order if o in seg.index])
        ax.bar(seg.index, seg.values, color=PALETTE[0], edgecolor='#0f172a')
    ax.set(title=title, xlabel=col, ylabel='MAE')
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('../outputs/error_by_segment.png', dpi=150, bbox_inches='tight')
plt.show()

## Evaluation Summary

| Metric | Value |
|---|---|
| R² | see output above |
| RMSE | see output above |
| MAE | see output above |
| MAPE | see output above |

**Observations:**
- Residuals should be roughly centred at zero (no systematic bias).
- Larger errors expected in extreme weather / spring months.
- Q-Q plot reveals whether errors are approximately normal.

**Next:** `09_Model_Interpretation.ipynb`
